In [1]:
# ==============================
# Hotel Booking Cancellation Prediction
# Step 1: Load Silver Data
# ==============================

import pandas as pd

SILVER_PATH = "../data/silver/hotel_bookings_silver.csv"

df_ml = pd.read_csv(SILVER_PATH)

print(df_ml.shape)
df_ml.head()

(86638, 42)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,ingestion_timestamp,source_file_name,batch_id,total_guests,arrival_date,total_nights,booking_status,estimated_revenue,silver_processed_timestamp,meal_plan
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,1.0,2015-07-01,1,Not Canceled,75.0,2026-05-16 16:13:00.384678,Bed and Breakfast
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,1.0,2015-07-01,1,Not Canceled,75.0,2026-05-16 16:13:00.384678,Bed and Breakfast
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,2.0,2015-07-01,2,Not Canceled,196.0,2026-05-16 16:13:00.384678,Bed and Breakfast
3,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,2.0,2015-07-01,2,Not Canceled,214.0,2026-05-16 16:13:00.384678,Bed and Breakfast
4,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,2.0,2015-07-01,2,Not Canceled,206.0,2026-05-16 16:13:00.384678,Full Board


In [ ]:
# check the distribution of the target variable
df_ml["is_canceled"].value_counts()

0    62652
1    23986
Name: is_canceled, dtype: int64

In [4]:
df_ml["is_canceled"].value_counts(normalize=True)*100

0    72.314689
1    27.685311
Name: is_canceled, dtype: float64

In [5]:
feature_columns = [
    "hotel",
    "lead_time",
    "arrival_date_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "reserved_room_type",
    "deposit_type",
    "customer_type",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "total_nights",
    "total_guests"
]

target_column = "is_canceled"

X = df_ml[feature_columns]
y = df_ml[target_column]

print(X.shape)
print(y.shape)

(86638, 23)
(86638,)


In [6]:
# ==============================
# Hotel Booking Cancellation Prediction
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Split categorical and numerical columns
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

Categorical features:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']

Numerical features:
['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests']

Number of categorical features: 9
Number of numerical features: 14


In [7]:
# ==============================
# Train-Test Split
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train shape: (69310, 23)
X_test shape: (17328, 23)
y_train shape: (69310,)
y_test shape: (17328,)

Training target distribution:
0    72.31424
1    27.68576
Name: is_canceled, dtype: float64

Testing target distribution:
0    72.316482
1    27.683518
Name: is_canceled, dtype: float64


In [8]:
# ==============================
# Preprocessing Pipeline
# ==============================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [9]:
# ==============================
# Baseline Model: Logistic Regression
# ==============================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]
)

log_reg_model.fit(X_train, y_train)

y_pred = log_reg_model.predict(X_test)
y_pred_proba = log_reg_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7435941828254847
Precision: 0.5252857142857142
Recall: 0.7665207421304983
F1 Score: 0.6233788251250317
ROC-AUC: 0.8299447056519761

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.73      0.81     12531
           1       0.53      0.77      0.62      4797

    accuracy                           0.74     17328
   macro avg       0.71      0.75      0.71     17328
weighted avg       0.79      0.74      0.76     17328


Confusion Matrix:
[[9208 3323]
 [1120 3677]]


In [10]:
# ==============================
# Step 3: Train Random Forest Model
# ==============================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Create Random Forest pipeline
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train model
random_forest_model.fit(X_train, y_train)

# Predict class labels
y_pred_rf = random_forest_model.predict(X_test)

# Predict cancellation probability
y_pred_proba_rf = random_forest_model.predict_proba(X_test)[:, 1]

# Evaluate model
print("Random Forest Model Performance")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Model Performance
Accuracy: 0.8146352723915051
Precision: 0.6379460400348129
Recall: 0.764019178653325
F1 Score: 0.6953139821665718
ROC-AUC: 0.8864404852160097

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.83      0.87     12531
           1       0.64      0.76      0.70      4797

    accuracy                           0.81     17328
   macro avg       0.77      0.80      0.78     17328
weighted avg       0.83      0.81      0.82     17328


Confusion Matrix:
[[10451  2080]
 [ 1132  3665]]


In [11]:
model_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_pred_proba),
        roc_auc_score(y_test, y_pred_proba_rf)
    ]
})

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.743594,0.525286,0.766521,0.623379,0.829945
1,Random Forest,0.814635,0.637946,0.764019,0.695314,0.886440


In [13]:

import sys
!{sys.executable} -m pip install xgboost

     -------------------------------------- 124.9/124.9 MB 5.5 MB/s eta 0:00:00


In [14]:
# ==============================
# Step 4: Train XGBoost Model
# ==============================

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train model
xgb_model.fit(X_train, y_train)

# Predict class labels
y_pred_xgb = xgb_model.predict(X_test)

# Predict cancellation probability
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate model
print("XGBoost Model Performance")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Model Performance
Accuracy: 0.8199445983379502
Precision: 0.7189866805954558
Recall: 0.5739003543881592
F1 Score: 0.6383028054718293
ROC-AUC: 0.8801610737911153

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.91      0.88     12531
           1       0.72      0.57      0.64      4797

    accuracy                           0.82     17328
   macro avg       0.78      0.74      0.76     17328
weighted avg       0.81      0.82      0.81     17328


Confusion Matrix:
[[11455  1076]
 [ 2044  2753]]


In [15]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_pred_proba),
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_xgb)
    ]
})

model_comparison.sort_values(by="ROC-AUC", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,Random Forest,0.814635,0.637946,0.764019,0.695314,0.886440
2,XGBoost,0.819945,0.718987,0.573900,0.638303,0.880161
0,Logistic Regression,0.743594,0.525286,0.766521,0.623379,0.829945
